# Model AUC comparison — time-to-platinum

Mean IPCW cumulative/dynamic AUC(t) on the held-out test fold for the three
multivariable models (elastic-net Cox, landmark XGBoost, Dynamic DeepHit),
shown as a **single grouped barplot** with bars paired across landmarks
(0d and +90d) so the cross-landmark comparison is visible at a glance.

Sources (one CSV per model × landmark):
- **Cox**     `cox/landmark_{n}/both/cox_agg_multivariable_metrics.csv`        → row with `endpoint == "platinum"`, column `test_mean_auc_t`
- **XGBoost** `xgboost/landmark_{n}/both/landmark_xgboost_metrics.csv`         → row with `endpoint == "platinum"`, column `mean_auc_t`
- **DeepHit** `deephit/landmark_{n}/PLATINUM/dynamic_deephit_metrics_PLATINUM.csv` → row with `event == "PLATINUM"`, column `mean_auc_t`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 130,
    "savefig.dpi": 220,
    "savefig.bbox": "tight",
    "font.size": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})


## Configure paths


In [ ]:
BASE = Path(
    "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/"
    "abstract_final_survival_analysis/local_runs"
)
OUT_DIR = Path("./figures")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Loaders + render function

One loader per model. Each pulls a single scalar `mean AUC(t)` from the
metrics CSV at the requested landmark.


In [ ]:
def load_cox(landmark):
    p = BASE / "cox" / f"landmark_{landmark}" / "both" / "cox_agg_multivariable_metrics.csv"
    df = pd.read_csv(p)
    row = df.loc[df["endpoint"] == "platinum"].iloc[0]
    return float(row["test_mean_auc_t"])


def load_xgb(landmark):
    p = BASE / "xgboost" / f"landmark_{landmark}" / "both" / "landmark_xgboost_metrics.csv"
    df = pd.read_csv(p)
    row = df.loc[df["endpoint"] == "platinum"].iloc[0]
    return float(row["mean_auc_t"])


def load_deephit(landmark):
    p = BASE / "deephit" / f"landmark_{landmark}" / "PLATINUM" / "dynamic_deephit_metrics_PLATINUM.csv"
    df = pd.read_csv(p)
    row = df.loc[df["event"] == "PLATINUM"].iloc[0]
    return float(row["mean_auc_t"])


MODEL_ORDER = ["Elastic-Net Cox", "XGBoost Survival", "Dynamic DeepHit"]
LOADERS = {
    "Elastic-Net Cox":  load_cox,
    "XGBoost Survival": load_xgb,
    "Dynamic DeepHit":  load_deephit,
}
MODEL_COLORS = {
    "Elastic-Net Cox":  "#4C72B0",  # steel blue
    "XGBoost Survival": "#B58900",  # deep gold (Solarized)
    "Dynamic DeepHit":  "#55A868",  # muted green
}
LANDMARKS = [0, 90]


def render_grouped(landmarks=LANDMARKS):
    # collect all values up front
    values = {m: [LOADERS[m](lm) for lm in landmarks] for m in MODEL_ORDER}
    n_models = len(MODEL_ORDER)
    bar_width = 0.24
    x_positions = np.arange(len(landmarks))
    offsets = (np.arange(n_models) - (n_models - 1) / 2) * bar_width

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for i, m in enumerate(MODEL_ORDER):
        bar_x = x_positions + offsets[i]
        bars = ax.bar(bar_x, values[m], bar_width,
                      color=MODEL_COLORS[m],
                      edgecolor="white", linewidth=0.8,
                      label=m)
        for x, v in zip(bar_x, values[m]):
            ax.text(x, v + 0.008, f"{v:.3f}",
                    ha="center", va="bottom",
                    fontsize=10, weight="semibold",
                    color=MODEL_COLORS[m])

    all_vals = [v for vs in values.values() for v in vs]
    y_top = min(1.0, max(all_vals) * 1.12)
    ax.set_ylim(0.45, y_top)

    ax.set_xticks(x_positions)
    ax.set_xticklabels([f"{('+' if lm > 0 else '')}{lm} days"
                        for lm in landmarks])
    ax.axhline(0.5, color="grey", linestyle=":", linewidth=0.9)
    ax.set_ylabel("Test Mean AUC(t)")
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="upper right", fontsize=10, ncol=1)
    return fig


## Render the grouped barplot


In [ ]:
fig = render_grouped()
for ext in ("png", "pdf"):
    out = OUT_DIR / f"auc_grouped_barplot_platinum.{ext}"
    fig.savefig(out)
    print(f"wrote {out}")
plt.show()
